# Aircraft classification

In [6]:
import os
import math
import random
from pathlib import Path
from typing import List,Tuple

import gdown
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import Tensor
import torch.nn as nn
import  torch.nn.functional as F
from torch.optim import SGD,Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau, OneCycleLR
from torch.utils.data import Dataset, DataLoader
import torchvision.datasets as datasets


import torchvision
from torchvision import transforms, models
from torchvision.transforms import RandAugment

In [7]:
# 1. Define an absolute Path on drive
drive_checkpoint_dir = '/content/drive/MyDrive/Aircaft classification/checkpoints'

# 2. Create a folder if doesn't exist
os.makedirs(drive_checkpoint_dir, exist_ok=True)

# 3. Define a path of the file on Drive
cnn_checkpoint1 = Path(drive_checkpoint_dir) / "cnn_checkpoint1.pt"

# (Optional) If the Id is useful, save it on Drive
cnn_checkpoint1_id = "aircraft_cnn_epoch_10_loss_0.0500.pth"


# Getting data

In [8]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Training transforms for Phase 1
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
   # transforms.ColorJitter(
       #  brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1
    #),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])
# Folder where the data is downloaded
from google.colab import drive
!rm -rf /content/drive
drive.mount('/content/drive')
data_dir = '/content/drive/MyDrive/Colab Notebooks/data_aircraft'

# Download the dataset
print("Downloading the dataset...")
train_dataset = datasets.FGVCAircraft(
    root=data_dir,
    split = 'train',
    annotation_level='variant',
    download=True,
    transform=train_transform
)
val_dataset = datasets.FGVCAircraft(
    root=data_dir,
    split = 'val',
    annotation_level='variant',
    download= True,
    transform= val_transform
)
# Download the test
print("Downloading the test dataset...")
test_dataset = datasets.FGVCAircraft(
    root=data_dir,
    split = 'test',
    annotation_level='variant',
    download= True,
    transform=test_transform
)

# --- Explore data
print(f"\n Number of samples in training: {len(train_dataset)}")
print(f"Size of the test data: {len(test_dataset)}")
print(f"Number of classes (variants): {len(train_dataset.classes)}")

image, label_idx = train_dataset[0]
class_name = train_dataset.classes[label_idx]

print(f"\nAnalysis of the first sample:")
print(f" - Dimension of the image tensor: {image.shape} (Channels, Heigth, Length)")
print(f" - Index of the label: {label_idx}")
print(f" - Name of the variant: {class_name}")

# ToTensor() converts the images in the format [C, H, W],
# but matplotlib wants images in this format [H, W, C], so I use permute()
plt.imshow(image.permute(1, 2, 0))
plt.title(f"Variant: {class_name}")
plt.axis('off')
plt.show()

^C


KeyboardInterrupt: 

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader_cnn = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [ ]:
def plot_training_history(training_dict,title_suffix =''):
    plt.figure(figsize=(10,8))
    plt.plot(training_dict['train_loss'],label = 'Training_loss')
    plt.plot(training_dict['val_loss'],label = 'Validation_loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'Training vs  Validation Loss {title_suffix}')
    plt.legend()
    plt.show()

    #Validation accuracy plot
    plt.figure(figsize= (10,8))
    plt.plot(training_dict['val_acc'],label= 'Validation accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy %')
    plt.title(f'Validation accuracy over epochs {title_suffix}')
    plt.legend()
    plt.show()

def mixup_data(x, y, alpha=0.2, device='cuda'):
    if alpha <= 0: return x, y, None, None, 1.0
    lam = torch.distributions.Beta(alpha, alpha).sample().item()
    batch_size = x.size()[0]
    if batch_size == 1: return x, y, None, None, 1.0
    index = torch.randperm(batch_size).to(device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, index, lam

def mixup_criterion(criterion, preds, y_a, y_b, lam):
    return lam * criterion(preds, y_a) + (1 - lam) * criterion(preds, y_b)

###
def accuracy(output,target):
    preds = output.argmax(dim=1)
    return (preds==target).sum().item()

def visualize_predictions(model,dataset,device,num_samples = 5,class_names = None):
    indices = random.sample(range(len(dataset)),num_samples)
    fig, axes = plt.subplots(1, num_samples, figsize=(4*num_samples, 4))

    for i,idx in enumerate(indices):
        img,label = dataset[idx]
        img_vis = img.clone()
         # Prediction
        input_tensor = img.unsqueeze(0).to_device()
        with torch.no_grad():
             output = model(input_tensor)
             pred_label = output.argmax(dim=1).item()
        # Map indices to class names if provided
        true_name = class_names[label] if class_names else str(label)
        pred_name = class_names[pred_label] if class_names else str(pred_label)
        axes[i].imshow(img_vis)
        axes[i].axis('off')
        axes[i].set_title(f"True: {true_name}\nPred: {pred_name}")
    plt.tight_layout()
    plt.show()




## Training function for CNN and ResNet-18

In [ ]:
def train_model(model,train_dataset,val_dataset,
                checkpoint_path,
                device=None,
                optimizer_type='adam',
                scheduler_type='ReduceLROnPlateau',
                lr_backbone=None,
                lr_head=None,
                num_epochs=100,
                early_stop_patience=10,
                mixup_alpha=0.0,
                steps_per_epoch=None,
                title_suffix=""):

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False)

    model.to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # Optimizer setup
    if optimizer_type.lower() == 'adam':
        optimizer = Adam(model.parameters(), lr=1e-4)
    elif optimizer_type.lower() == 'sgd':
        optimizer = SGD(model.parameters(),weight_decay= 1e-4)
    else:
        print("The optimizer must be Adam or SGD")
    # Scheduler
    if scheduler_type == 'ReduceLROnPlateau':
        scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0
    no_improve_epochs = 0
    for epoch in range(1, num_epochs+1):
        model.train()
        running_loss, running_correct, running_total = 0, 0, 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if mixup_alpha > 0:
                imgs, y_a, y_b, _, lam = mixup_data(imgs, labels, alpha=mixup_alpha, device=device)
            else:
                y_a, y_b, lam = labels, None, 1.0

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = mixup_criterion(criterion, outputs, y_a, y_b, lam) if mixup_alpha > 0 else criterion(outputs, y_a)
            loss.backward()
            optimizer.step()
            if scheduler_type == 'OneCycleLR':
                scheduler.step()

            running_loss += loss.item() * imgs.size(0)
            running_correct += accuracy(outputs, y_a)
            running_total += imgs.size(0)

        train_loss = running_loss / running_total
        train_acc  = 100 * running_correct / running_total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                val_correct += accuracy(outputs, labels)
                val_total += imgs.size(0)

        val_loss /= val_total
        val_acc = 100 * val_correct / val_total

        if scheduler_type == 'ReduceLROnPlateau':
            scheduler.step(val_acc)

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            no_improve_epochs = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
                'best_val_acc': best_val_acc,
                'history': history
            }, checkpoint_path)
            print(f"Epoch {epoch}: New best model saved with Val Acc = {best_val_acc:.2f}%")
        else:
            no_improve_epochs += 1

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch} | Train Loss {train_loss:.4f} | Train Acc {train_acc:.2f}% | "
              f"Val Loss {val_loss:.4f} | Val Acc {val_acc:.2f}%")

        if no_improve_epochs >= early_stop_patience:
            print(f"No improvement for {early_stop_patience} epochs. Early stopping.")
            break
    print(f"Training finished. Best val accuracy: {best_val_acc:.2f}%")
    print("Checkpoint saved at:", checkpoint_path)
    plot_training_history(history, title_suffix=title_suffix)
    return history, best_val_acc


In [ ]:
def evaluate_testdata(model, data_loader, label_smoothing=0.0, verbose=True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for imgs, labels in data_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = running_loss / len(data_loader)
    accuracy = 100.0 * correct / total

    if verbose:
        print(f"Evaluation -> Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

    return avg_loss, accuracy

In [ ]:

def evaluate_model( model_class=None,checkpoint_path=None,test_loader=None,test_dataset=None,num_classes=100,
    label_smoothing=0.1,model_type='cnn',verbose_name='Model',device=None,visualize=False,num_samples=5):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --- Load model ---
    if model_type == 'cnn':
        model = model_class(num_classes=num_classes)
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
    elif model_type == 'resnet':
        model = models.resnet18(weights=None)
        in_features = model.fc.in_features
        model.fc = torch.nn.Linear(in_features, num_classes)
        checkpoint = torch.load(checkpoint_path, map_location=device)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint["model_state_dict"])
        else:
            model.load_state_dict(checkpoint)
    else:
        raise ValueError("model_type must be 'cnn' or 'resnet'")

    model.to(device)
    model.eval()

    # --- Evaluate ---
    test_loss, test_acc = evaluate_testdata(model, test_loader, label_smoothing=label_smoothing, verbose=False)

    # --- Print clean summary ---
    print(f"{verbose_name} Evaluation:")
    print(f"Test Loss : {test_loss:.4f}")
    print(f"Test Acc  : {test_acc:.2f}%")
    print("-"*40)

    # --- Optional visualization ---
    if visualize and test_dataset is not None:
        visualize_predictions(model, test_dataset, device, num_samples=num_samples)

    return test_loss, test_acc


## Part 1: design your own network

Your goal is to implement a convolutional neural network for image classification and train it from scratch on `FGVCAircraft`. You should consider yourselves satisfied once you obtain a classification accuracy on the test split of ~50%. You are free to achieve this however you want, except for a few rules you must follow:

- Compile this notebook by displaying the results obtained by the best model you found throughout your experimentation; then show how, by removing some of its components, its performance drops. In other words, do an *ablation study* to prove that your design choices have a positive impact on the final result.

- Do not instantiate an off-the-self PyTorch network. Instead, construct your network as a composition of existing PyTorch layers. In more concrete terms, you can use e.g. `torch.nn.Linear`, but you cannot use e.g. `torchvision.models.alexnet`.

- Show your results and ablations with plots, tables, images, etc. — the clearer, the better.

Don't be too concerned with your model performance: the ~50% is just to give you an idea of when to stop. Keep in mind that a thoroughly justified model with lower accuracy will be rewarded more points than a poorly experimentally validated model with higher accuracy.

In [ ]:
class AircraftCNN(nn.Module):
    def __init__(self, num_classes=100):
        super(AircraftCNN, self).__init__()

        # --- BLOCK 1 (2 Conv) ---
        self.conv1a = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1a = nn.BatchNorm2d(64)
        self.conv1b = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn1b = nn.BatchNorm2d(64)

        # --- BLOCK 2 (2 Conv) ---
        self.conv2a = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2a = nn.BatchNorm2d(128)
        self.conv2b = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn2b = nn.BatchNorm2d(128)

        # --- BLOCK 3 (2 Conv) ---
        self.conv3a = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3a = nn.BatchNorm2d(256)
        self.conv3b = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn3b = nn.BatchNorm2d(256)

        # --- BLOCK 4 (2 Conv) ---
        self.conv4a = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn4a = nn.BatchNorm2d(512)
        self.conv4b = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.bn4b = nn.BatchNorm2d(512)

        # Pooling e Dropout
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        #
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

        self.dropout_conv = nn.Dropout2d(0.2)
        self.dropout_fc = nn.Dropout(0.3)

        #
        #
        self.fc1 = nn.Linear(512 * 4 * 4, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, num_classes)

    def forward(self, x):
        # Block 1
        x = F.relu(self.bn1a(self.conv1a(x)))
        x = self.pool(F.relu(self.bn1b(self.conv1b(x))))
        x = self.dropout_conv(x)

        # Block 2
        x = F.relu(self.bn2a(self.conv2a(x)))
        x = self.pool(F.relu(self.bn2b(self.conv2b(x))))
        x = self.dropout_conv(x)

        # Block 3
        x = F.relu(self.bn3a(self.conv3a(x)))
        x = self.pool(F.relu(self.bn3b(self.conv3b(x))))

        # Block 4
        x = F.relu(self.bn4a(self.conv4a(x)))
        x = self.pool(F.relu(self.bn4b(self.conv4b(x))))
        x = self.dropout_conv(x)

        # Adaptive Pooling
        x = self.adaptive_pool(x)

        # Flatten
        x = x.view(x.shape[0], -1)

        # FC layers
        x = F.relu(self.fc1(x))
        x = self.dropout_fc(x)

        x = F.relu(self.fc2(x))
        x = self.dropout_fc(x)

        x = self.fc3(x)

        return x

## Model training

In [ ]:
train_model(
    model=AircraftCNN(num_classes= 100),
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    checkpoint_path=cnn_checkpoint1,
    optimizer_type='adam',
    scheduler_type='ReduceLROnPlateau',
    num_epochs=150,
    early_stop_patience=15,
    mixup_alpha=0.0,
    title_suffix="- CNN from scratch - Phase 1"
)

In [ ]:
## Model
test_loss_cnn_phase1, test_acc_cnn_phase1 = evaluate_model(
    model_class=AircraftCNN,
    checkpoint_path=cnn_checkpoint1,
    test_loader=test_loader_cnn,
    test_dataset=test_dataset,
    model_type='cnn',
    verbose_name="CNN Phase 1",
    visualize=True,
    num_samples=5
)

## Ablation study

In [ ]:
class ManualCNN_ablation(AircraftCNN):
    def __init__(self, num_classes=100,ablation = None, input_size = 224):
      # ablation options
      # - remove block 4
      # - remove adaptive pool
      # - remove bn
      # - remove dropout
        super().__init__(num_classes)
        self.ablation = ablation

        if self.ablation == "remove block 4":
          self.conv4a = self.conv4b = self.bn4a = self.bn4b = nn.Identity()
        elif self.ablation == 'remove adaptive pool':
          self.adaptive_pool = nn.Identity()
          fc1_input_size = 512*14*14
          self.fc1 = nn.Linear(fc1_input_size,1024)
        elif self.ablation == "remove bn":
          self.bn1a = self.bn1b = nn.Identity()
          self.bn2a = self.bn2b = nn.Identity()
          self.bn3a = self.bn3b = nn.Identity()
          self.bn4a = self.bn4b = nn.Identity()
        elif self.ablation =='remove dropout':
          self.dropout_conv = nn.Idendity()
          self.dropout_fc = nn.Identity()
    def forward(self, x):
        # Block 1
        x = F.relu(self.bn1a(self.conv1a(x)))
        x = self.pool(F.relu(self.bn1b(self.conv1b(x))))
        if self.ablation != 'remove dropout': x = self.dropout_conv(x)

        # Block 2
        x = F.relu(self.bn2a(self.conv2a(x)))
        x = self.pool(F.relu(self.bn2b(self.conv2b(x))))
        if self.ablation != 'remove dropout':
           x = self.dropout_conv(x)

        # Block 3
        x = F.relu(self.bn3a(self.conv3a(x)))
        x = self.pool(F.relu(self.bn3b(self.conv3b(x))))

        # Block 4 -o
        if self.ablation == "remove block 4":
            pass
        else:
            x = F.relu(self.bn4a(self.conv4a(x)))
            x = self.pool(F.relu(self.bn4b(self.conv4b(x))))
            if self.ablation != 'remove dropout': x = self.dropout_conv(x)

        # Adaptive Pooling if it is Identity nothing happens
        x = self.adaptive_pool(x)

        # Flatten
        x = x.view(x.shape[0], -1)

        # FC layers
        x = F.relu(self.fc1(x))
        if self.ablation != 'remove dropout':
           x = self.dropout_fc(x)

        x = F.relu(self.fc2(x))
        if self.ablation != 'remove dropout':
          x = self.dropout_fc(x)

        x = self.fc3(x)

        return x


In [ ]:
#Defining list to store all the study results
ablation_results_list = []


# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Initialize model ---
baseline_model = ManualCNN_ablation(num_classes=37, dropout_rate=0.2)
baseline_model.to(device)

# --- Load checkpoint ---
checkpoint = torch.load(cnn_checkpoint1, map_location=device)
baseline_model.load_state_dict(checkpoint['model_state_dict'])

# --- Evaluate on test set ---
test_loss, test_acc = evaluate_testdata(baseline_model, test_loader_cnn, label_smoothing=0.1)

# --- Store results ---
baseline_results = {'name': 'Baseline', 'loss': test_loss, 'accuracy': test_acc}

ablation_results_list.append(baseline_results)

##  REMOVE BLOCK 4

In [ ]:
# --- Initialize Ablation 1 model ---
ablation1_model = ManualCNN_ablation(num_classes=100, ablation="remove_block4")

# --- Train the model ---
history_ablation1, checkpoint_ablation1 = train_model(
    ablation1_model, model_name="cnn_ablation1"
)

# --- Load best checkpoint and evaluate ---
checkpoint = torch.load(checkpoint_ablation1, map_location=device)
ablation1_model.load_state_dict(checkpoint['model_state_dict'])

loss1, acc1 = evaluate_testdata(ablation1_model, test_loader_cnn, label_smoothing=0.1)

# --- Store results ---
ablation1_results = {'name': 'Ablation 1: Remove Block 4', 'loss': loss1, 'accuracy': acc1}
ablation_results_list.append(ablation1_results)